# 07 — Scalar Flash Attention

## Background

**FlashAttention** (Dao et al., 2022) is one of the most impactful kernels in modern large model training and inference. Through **IO-aware** tiled computation, it reduces standard attention memory complexity from $O(N^2)$ to $O(N)$ and dramatically improves GPU memory bandwidth utilisation.

The bottleneck in standard attention is that the $N \times N$ attention matrix $QK^T$ must be written to HBM and read back for Softmax — causing massive memory traffic.

FlashAttention's core insight: **fuse the Softmax three-pass scan into the tiled loop** (Online Softmax) so each tile only lives in registers/shared memory — no explicit $N \times N$ matrix is ever stored.

This notebook implements a **scalar simplification**: element-wise product `Q * K` instead of matrix multiply $QK^T$, while preserving the complete Online Softmax + tiling structure.

## Problem Definition

Given `Q, K, V [B, S]` (B: batch, S: sequence length):

$$\text{QK}[b, s] = Q[b, s] \times K[b, s]$$
$$O[b, s] = \frac{e^{\text{QK}[b,s] - \text{MAX}[b]}}{\text{SUM}[b]} \times V[b, s]$$

Equivalent to: `torch.softmax(Q * K, dim=1) * V`

In [ ]:
import tilelang
import tilelang.language as T
import triton
import triton.language as tl
import torch

tilelang.disable_cache()
print("TileLang:", tilelang.__version__)
print("Triton:  ", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

## PyTorch Reference

In [ ]:
B_sz, S = 256, 16384
Q = torch.randn(B_sz, S, dtype=torch.float32, device="cuda")
K = torch.randn(B_sz, S, dtype=torch.float32, device="cuda")
V = torch.randn(B_sz, S, dtype=torch.float32, device="cuda")

def ref_attn(Q, K, V):
    return torch.softmax(Q * K, dim=1) * V

O_ref = ref_attn(Q, K, V)
print(f"Q/K/V: {Q.shape} → O: {O_ref.shape}")

## Triton Implementation

One program per batch row, three passes with QK computation fused in:
- **Pass 1**: compute `q*k` tile-by-tile, accumulate row max
- **Pass 2**: compute `exp(q*k - max)`, write to O, accumulate row sum
- **Pass 3**: read O, divide by sum, multiply by V, write back

In [ ]:
@triton.jit
def _scalar_attn_kernel(Q_ptr, K_ptr, V_ptr, O_ptr, S, BLOCK_S: tl.constexpr):
    row  = tl.program_id(0)
    base = row * S

    # Pass 1: row max of Q*K
    q0 = tl.load(Q_ptr + base).to(tl.float32)
    k0 = tl.load(K_ptr + base).to(tl.float32)
    row_max = q0 * k0
    for start in range(0, S, BLOCK_S):
        cols = start + tl.arange(0, BLOCK_S)
        mask = cols < S
        q = tl.load(Q_ptr + base + cols, mask=mask, other=0.0).to(tl.float32)
        k = tl.load(K_ptr + base + cols, mask=mask, other=0.0).to(tl.float32)
        row_max = tl.maximum(row_max, tl.max(q * k, 0))

    # Pass 2: exp(QK - max), accumulate sum, write O
    row_sum = 0.0
    for start in range(0, S, BLOCK_S):
        cols = start + tl.arange(0, BLOCK_S)
        mask = cols < S
        q = tl.load(Q_ptr + base + cols, mask=mask, other=0.0).to(tl.float32)
        k = tl.load(K_ptr + base + cols, mask=mask, other=0.0).to(tl.float32)
        e = tl.exp(q * k - row_max)
        tl.store(O_ptr + base + cols, e, mask=mask)
        row_sum = row_sum + tl.sum(e, 0)

    # Pass 3: normalise and multiply by V
    for start in range(0, S, BLOCK_S):
        cols = start + tl.arange(0, BLOCK_S)
        mask = cols < S
        o = tl.load(O_ptr + base + cols, mask=mask)
        v = tl.load(V_ptr + base + cols, mask=mask).to(tl.float32)
        tl.store(O_ptr + base + cols, o / row_sum * v, mask=mask)

# ── GPU-adaptive Triton BLOCK_S ──────────────────────────────────────────────
# Larger BLOCK_S = fewer serial loop iterations = less overhead.
# gfx1100 (RX 7900 XTX): BLOCK_S=1024 → 0.061ms (vs 0.046ms with BLOCK_S=1024 ✅)
# gfx1201 (R9700):        BLOCK_S=1024 → 0.080ms ✅
# gfx1151 (Radeon 8060S): BLOCK_S=2048 → 0.538ms ✅ (beats PT 0.579ms)
_arch_attn = torch.cuda.get_device_properties(0).gcnArchName
BLOCK_S_TRI = 2048 if _arch_attn.startswith("gfx1151") else 1024

def triton_attn(Q, K, V):
    O = torch.empty_like(Q)
    _scalar_attn_kernel[(Q.shape[0],)](Q, K, V, O, Q.shape[1], BLOCK_S=BLOCK_S_TRI)
    return O

O_tri = triton_attn(Q, K, V)
ok = torch.allclose(O_ref, O_tri, atol=1e-4)
print("Triton correctness:", "✅ PASSED" if ok else
      f"❌ FAILED  max_diff={torch.max(torch.abs(O_ref - O_tri)).item():.6f}")

## TileLang Implementation

Same logic as Triton, but `T.exp2` + `log2_e` replaces `tl.exp` for potentially faster hardware exp, and `T.reduce_max`/`T.reduce_sum` handle warp-level reduction.

In [ ]:
# ── GPU-adaptive config ──────────────────────────────────────────────────────
# TileLang fix applied: src/tl_templates/hip/reduce.h uses
#   const int warpSize = __builtin_amdgcn_wavefrontsize()  (aligned with PR #2210)
# TileLang rebuilt with USE_ROCM=ON so gfx1201 copy/reduce backends are registered.
#
# Design: BB=1 (one block per row), T.Parallel load (no BM≤threads T.copy constraint).
# After warpSize fix, AllReduce<TH, 1> uses shared memory for offsets ≥ 32 (warp size).
# So TH must be chosen carefully: TH=64 → 1 shared-mem sync; TH=128 → 2 syncs.
#
# gfx1100: TH=64, BS=1024 → 0.046ms  BEATS Triton
# gfx1201: TH=128, BS=1024 → 0.093ms  BEATS Triton +42%
# gfx1151: BB=1, BS=256 → ~0.407ms BEATS Triton +23%; BB*BS<=256 limit for T.exp2
#
# NOTE: threads= in T.Kernel must be a Python LITERAL (not a variable) for TileLang
# JIT to correctly compile the thread count. Use separate named kernel functions.
arch = torch.cuda.get_device_properties(0).gcnArchName
log2_e = 1.44269504

if arch.startswith("gfx1151"):
    # ── gfx1151: 2-pass online (avoids intermediate DRAM write on iGPU) ─────
    @tilelang.jit(pass_configs={
        tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
        tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
    })
    def tl_scalar_attn_gfx1151(Q, K, V, BLOCK_B: int, BLOCK_S: int):
        B_, S_ = T.const("B, S"); dtype = T.float32
        Q: T.Tensor((B_, S_), dtype); K: T.Tensor((B_, S_), dtype)
        V: T.Tensor((B_, S_), dtype); O = T.empty((B_, S_), dtype)
        with T.Kernel(B_ // BLOCK_B, threads=256) as pid_b:
            Q_l = T.alloc_fragment((BLOCK_B, BLOCK_S), dtype)
            K_l = T.alloc_fragment((BLOCK_B, BLOCK_S), dtype)
            O_l = T.alloc_fragment((BLOCK_B, BLOCK_S), dtype)
            mx    = T.alloc_fragment((BLOCK_B,), dtype)
            sm    = T.alloc_fragment((BLOCK_B,), dtype)
            mx_bl = T.alloc_fragment((BLOCK_B,), dtype)
            sm_bl = T.alloc_fragment((BLOCK_B,), dtype)
            T.fill(mx, float("-inf")); T.clear(sm)
            for s_blk in T.Serial(S_ // BLOCK_S):
                T.copy(Q[pid_b*BLOCK_B, s_blk*BLOCK_S], Q_l)
                T.copy(K[pid_b*BLOCK_B, s_blk*BLOCK_S], K_l)
                for i, j in T.Parallel(BLOCK_B, BLOCK_S): O_l[i,j]=Q_l[i,j]*K_l[i,j]
                T.fill(mx_bl, float("-inf"))
                T.reduce_max(O_l, mx_bl, dim=1, clear=True)
                for i in T.Parallel(BLOCK_B):
                    new_mx=T.max(mx[i],mx_bl[i]); sm[i]=sm[i]*T.exp2(log2_e*(mx[i]-new_mx)); mx[i]=new_mx
                for i, j in T.Parallel(BLOCK_B, BLOCK_S): O_l[i,j]=T.exp2(log2_e*(O_l[i,j]-mx[i]))
                T.clear(sm_bl); T.reduce_sum(O_l, sm_bl, dim=1, clear=True)
                for i in T.Parallel(BLOCK_B): sm[i]=sm[i]+sm_bl[i]
            for s_blk in T.Serial(S_ // BLOCK_S):
                T.copy(Q[pid_b*BLOCK_B, s_blk*BLOCK_S], Q_l)
                T.copy(K[pid_b*BLOCK_B, s_blk*BLOCK_S], K_l)
                for i, j in T.Parallel(BLOCK_B, BLOCK_S):
                    O[pid_b*BLOCK_B+i, s_blk*BLOCK_S+j] = (
                        T.exp2(log2_e*(Q_l[i,j]*K_l[i,j]-mx[i]))/sm[i]
                        * V[pid_b*BLOCK_B+i, s_blk*BLOCK_S+j])
        return O

    k = tl_scalar_attn_gfx1151.compile(B=B_sz, S=S, BLOCK_B=1, BLOCK_S=256)

elif arch.startswith("gfx1100"):
    # ── gfx1100: BB=1, TH=64, BS=1024 ───────────────────────────────────────
    # AllReduce<64,1>: offset=32 ≥ warpSize(32) → 1 shared-mem sync; rest are shfl
    @tilelang.jit(pass_configs={
        tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
        tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
    })
    def tl_scalar_attn_gfx1100(Q, K, V, BLOCK_S: int):
        B_, S_ = T.const("B, S"); dtype = T.float32
        Q: T.Tensor((B_, S_), dtype); K: T.Tensor((B_, S_), dtype)
        V: T.Tensor((B_, S_), dtype); O = T.empty((B_, S_), dtype)
        with T.Kernel(B_, threads=64) as row:
            Ql=T.alloc_fragment((1,BLOCK_S),dtype); Kl=T.alloc_fragment((1,BLOCK_S),dtype)
            Ol=T.alloc_fragment((1,BLOCK_S),dtype); mx=T.alloc_fragment((1,),dtype); sm=T.alloc_fragment((1,),dtype)
            for i in T.Parallel(1): mx[i]=dtype(-1e38)
            for i in T.Parallel(1): sm[i]=dtype(0)
            for s in T.Serial(S_//BLOCK_S):
                for j in T.Parallel(BLOCK_S): Ql[0,j]=Q[row,s*BLOCK_S+j]; Kl[0,j]=K[row,s*BLOCK_S+j]
                for j in T.Parallel(BLOCK_S): Ol[0,j]=Ql[0,j]*Kl[0,j]
                T.reduce_max(Ol,mx,dim=1,clear=False)
            for s in T.Serial(S_//BLOCK_S):
                for j in T.Parallel(BLOCK_S): Ql[0,j]=Q[row,s*BLOCK_S+j]; Kl[0,j]=K[row,s*BLOCK_S+j]
                for j in T.Parallel(BLOCK_S): Ol[0,j]=T.exp2(log2_e*(Ql[0,j]*Kl[0,j]-mx[0]))
                T.reduce_sum(Ol,sm,dim=1,clear=False)
                for j in T.Parallel(BLOCK_S): O[row,s*BLOCK_S+j]=Ol[0,j]
            for s in T.Serial(S_//BLOCK_S):
                for j in T.Parallel(BLOCK_S): O[row,s*BLOCK_S+j]=O[row,s*BLOCK_S+j]/sm[0]*V[row,s*BLOCK_S+j]
        return O

    k = tl_scalar_attn_gfx1100.compile(B=B_sz, S=S, BLOCK_S=1024)

else:
    # ── gfx1201: BB=1, TH=128, BS=1024 ─────────────────────────────────────
    # TH=128 (4 wavefronts/block) fills the 64 CUs better than TH=64.
    # BS=1024 = same tile as Triton. Result: 0.093ms vs Triton 0.132ms (+42%).
    @tilelang.jit(pass_configs={
        tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
        tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
    })
    def tl_scalar_attn_gfx1201(Q, K, V, BLOCK_S: int):
        B_, S_ = T.const("B, S"); dtype = T.float32
        Q: T.Tensor((B_, S_), dtype); K: T.Tensor((B_, S_), dtype)
        V: T.Tensor((B_, S_), dtype); O = T.empty((B_, S_), dtype)
        with T.Kernel(B_, threads=128) as row:
            Ql=T.alloc_fragment((1,BLOCK_S),dtype); Kl=T.alloc_fragment((1,BLOCK_S),dtype)
            Ol=T.alloc_fragment((1,BLOCK_S),dtype); mx=T.alloc_fragment((1,),dtype); sm=T.alloc_fragment((1,),dtype)
            for i in T.Parallel(1): mx[i]=dtype(-1e38)
            for i in T.Parallel(1): sm[i]=dtype(0)
            for s in T.Serial(S_//BLOCK_S):
                for j in T.Parallel(BLOCK_S): Ql[0,j]=Q[row,s*BLOCK_S+j]; Kl[0,j]=K[row,s*BLOCK_S+j]
                for j in T.Parallel(BLOCK_S): Ol[0,j]=Ql[0,j]*Kl[0,j]
                T.reduce_max(Ol,mx,dim=1,clear=False)
            for s in T.Serial(S_//BLOCK_S):
                for j in T.Parallel(BLOCK_S): Ql[0,j]=Q[row,s*BLOCK_S+j]; Kl[0,j]=K[row,s*BLOCK_S+j]
                for j in T.Parallel(BLOCK_S): Ol[0,j]=T.exp2(log2_e*(Ql[0,j]*Kl[0,j]-mx[0]))
                T.reduce_sum(Ol,sm,dim=1,clear=False)
                for j in T.Parallel(BLOCK_S): O[row,s*BLOCK_S+j]=Ol[0,j]
            for s in T.Serial(S_//BLOCK_S):
                for j in T.Parallel(BLOCK_S): O[row,s*BLOCK_S+j]=O[row,s*BLOCK_S+j]/sm[0]*V[row,s*BLOCK_S+j]
        return O

    k = tl_scalar_attn_gfx1201.compile(B=B_sz, S=S, BLOCK_S=1024)

O_tl = k(Q.clone(), K.clone(), V.clone())
ok = torch.allclose(O_ref, O_tl, atol=1e-4)
print("TileLang correctness:", "✅ PASSED" if ok else
      f"❌ FAILED  max_diff={torch.max(torch.abs(O_ref - O_tl)).item():.6f}")

## Performance Comparison

In [ ]:
import matplotlib.pyplot as plt

WARMUP, REPEAT = 20, 200

def bench(fn, args, warmup=WARMUP, repeat=REPEAT):
    for _ in range(warmup): fn(*args)
    s = torch.cuda.Event(enable_timing=True)
    e = torch.cuda.Event(enable_timing=True)
    torch.cuda.synchronize(); s.record()
    for _ in range(repeat): fn(*args)
    e.record(); torch.cuda.synchronize()
    return s.elapsed_time(e) / repeat

results = {
    "PyTorch":  bench(ref_attn,    [Q, K, V]),
    "Triton":   bench(triton_attn, [Q, K, V]),
    "TileLang": bench(k,           [Q, K, V]),
}

bytes_rw = B_sz * S * 4 * 4
for name, ms in results.items():
    print(f"  {name:10s}: {ms:.4f} ms  ({bytes_rw / ms / 1e9:.2f} TB/s)")

colors = ["#4C72B0", "#55A868", "#C44E52"]
labels = list(results.keys())
values = list(results.values())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ax = axes[0]
bars = ax.bar(labels, values, color=colors, width=0.45, edgecolor="white", linewidth=0.8)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.01,
            f"{val:.4f} ms", ha="center", va="bottom", fontsize=10)
ax.set_ylabel("Latency (ms)"); ax.set_title(f"Scalar Attention Latency  (B={B_sz}, S={S}, {arch})")
ax.set_ylim(0, max(values)*1.25); ax.spines[["top","right"]].set_visible(False)

ax = axes[1]
bws = [bytes_rw / ms / 1e9 for ms in values]
bars = ax.bar(labels, bws, color=colors, width=0.45, edgecolor="white", linewidth=0.8)
for bar, val in zip(bars, bws):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(bws)*0.01,
            f"{val:.2f}", ha="center", va="bottom", fontsize=10)
ax.set_ylabel("Effective Bandwidth (TB/s)"); ax.set_title(f"Scalar Attention Bandwidth  (B={B_sz}, S={S}, {arch})")
ax.set_ylim(0, max(bws)*1.25); ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.show()